# 🌾 FarmAI — Pipeline RAG multilingue (Hausa / Fulfulde)

**Auteur :** Aureon XY · **Plateforme :** Kaggle (GPU T4 recommandé)

---

## Objectif

Construire un assistant agricole vocal qui répond aux producteurs de tomate en **Hausa** ou **Fulfulde** à partir :

1. d'une **image** de feuille (détection de maladie via EfficientNet),
2. d'une **question** posée à l'oral ou à l'écrit,
3. d'une **base de connaissance** indexée (PDFs FarmAI HA/FF).

## Architecture

```
┌──────────────────┐   ┌─────────────────┐   ┌──────────────────┐
│  Image feuille   │──▶│  Vision FarmAI   │──▶│ Maladie détectée │
│  (.jpg / .png)   │   │ (EfficientNet)   │   │ ex: Late_blight  │
└──────────────────┘   └──────────────────┘   └────────┬─────────┘
                                                       │
┌──────────────────┐   ┌─────────────────┐             │
│ Question audio   │──▶│   ASR Waxal     │─┐           │
│ HA / FF          │   │   (optionnel)   │ │           │
└──────────────────┘   └─────────────────┘ │           │
                                           ▼           ▼
                                   ┌─────────────────────────┐
                                   │   Encodeur Serengeti    │
                                   │ (vectorise la requête)  │
                                   └────────────┬────────────┘
                                                ▼
                                   ┌─────────────────────────┐
                                   │   Index FAISS           │
                                   │ (chunks PDFs HA/FF)     │
                                   └────────────┬────────────┘
                                                ▼
                                   ┌─────────────────────────┐
                                   │   LLM générateur        │
                                   │ (réponse en HA ou FF)   │
                                   └────────────┬────────────┘
                                                ▼
                                   ┌─────────────────────────┐
                                   │   TTS Waxal (option)    │
                                   │   → audio final         │
                                   └─────────────────────────┘
```

## ⚠ Avant de lancer

1. **Active le GPU** : Settings → Accelerator → GPU T4 x2 (ou P100).
2. **Active Internet** : Settings → Internet → ON (nécessaire pour télécharger les modèles).
3. **Upload tes données** dans `Add Input` :
   - les 2 PDFs : `farmai_phrases_hausa.pdf`, `farmai_phrases_fulfulde.pdf`
   - (optionnel) ton modèle EfficientNet si tu veux brancher la vision
   - (optionnel) un fichier audio test pour l'ASR

Les chemins par défaut supposent que tu as ajouté un dataset Kaggle nommé `farmai-pdfs` contenant les 2 PDFs. Adapte sinon.

## 0 · Vérifications environnement

In [1]:
# --- Vérification GPU et environnement Kaggle ---
import sys, os, subprocess, platform

print(f"Python      : {sys.version.split()[0]}")
print(f"Plateforme  : {platform.system()} {platform.release()}")

try:
    import torch
    print(f"PyTorch     : {torch.__version__}")
    print(f"CUDA dispo  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU         : {torch.cuda.get_device_name(0)}")
        print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")
except ImportError:
    print("PyTorch     : non installé")

# Détection de l'environnement
on_kaggle = os.path.exists("/kaggle")
on_colab  = "google.colab" in sys.modules
print(f"Kaggle      : {on_kaggle}")
print(f"Colab       : {on_colab}")


Python      : 3.12.12
Plateforme  : Linux 6.6.122+
PyTorch     : 2.10.0+cu128
CUDA dispo  : True
GPU         : Tesla T4
VRAM        : 15.6 Go
Kaggle      : True
Colab       : False


## 1 · Installation des dépendances

On installe :
- **`pdfplumber`** pour extraire le texte des PDFs
- **`sentence-transformers`** pour Serengeti (encodeur multilingue africain)
- **`faiss-cpu`** comme base vectorielle (CPU suffit pour ~500 chunks)
- **`transformers` / `accelerate`** pour le LLM générateur
- **`bitsandbytes`** pour charger le LLM en 4-bit (économie VRAM)

> Sur Kaggle, certains packages sont déjà présents. Le `-q` rend l'output sobre.

In [2]:
!pip install -q --upgrade pip
!pip install -q pdfplumber sentence-transformers faiss-cpu
!pip install -q transformers accelerate bitsandbytes
!pip install -q langdetect  # détection auto de la langue de la question


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2 · Configuration des chemins

Adapte les chemins selon comment tu as ajouté tes données dans Kaggle.

> **Astuce Kaggle** : tes datasets uploadés apparaissent sous `/kaggle/input/<nom-du-dataset>/`. Tu peux explorer le panneau de droite pour copier le bon chemin.

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("aci")


In [4]:
from pathlib import Path

# --- Chemins d'entrée ---
# Adapte ces chemins selon ton dataset Kaggle
KAGGLE_INPUT = Path("/kaggle/input")

# Recherche automatique des PDFs dans tous les datasets Kaggle ajoutés
def find_pdf(name_substring):
    if not KAGGLE_INPUT.exists():
        # Fallback local
        local = Path(f"/mnt/user-data/uploads/{name_substring}")
        return local if local.exists() else None
    for p in KAGGLE_INPUT.rglob("*.pdf"):
        if name_substring.lower() in p.name.lower():
            return p
    return None

PDF_HAUSA = find_pdf("hausa")
PDF_FULFULDE = find_pdf("fulfulde")
PDF_FRENCH = find_pdf("farmai_output")  # adapte selon le nom de ton PDF (farmai_output.pdf)
PDF_ENG=find_pdf("farmai_output_english")
print(f"PDF Hausa     : {PDF_HAUSA}")
print(f"PDF Fulfulde  : {PDF_FULFULDE}")
print(f"PDF Français  : {PDF_FRENCH}")
print(f"PDF anglais  : {PDF_ENG}")
# --- Dossier de travail (toujours writable sur Kaggle) ---
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./farmai_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

INDEX_DIR = WORK_DIR / "faiss_index"
INDEX_DIR.mkdir(exist_ok=True)

print(f"Dossier travail : {WORK_DIR}")

assert PDF_HAUSA and PDF_HAUSA.exists(), "❌ PDF Hausa introuvable. Ajoute-le en Input."
assert PDF_FULFULDE and PDF_FULFULDE.exists(), "❌ PDF Fulfulde introuvable. Ajoute-le en Input."
assert PDF_FRENCH and PDF_FRENCH.exists(), "❌ PDF Français introuvable."
assert PDF_ENG and PDF_ENG.exists(), "❌ PDF Anglais introuvable."
print("✅ PDFs trouvés.")


PDF Hausa     : /kaggle/input/datasets/ngongajacquy/farmai-hausa-fulfulbe/farmai_phrases_hausa.pdf
PDF Fulfulde  : /kaggle/input/datasets/ngongajacquy/farmai-hausa-fulfulbe/farmai_phrases_fulfulde.pdf
PDF Français  : /kaggle/input/datasets/ngongajacquy/farmai-output-eng/farmai_output_english.pdf
PDF anglais  : /kaggle/input/datasets/ngongajacquy/farmai-output-eng/farmai_output_english.pdf
Dossier travail : /kaggle/working
✅ PDFs trouvés.


## 3 · Extraction et structuration des PDFs

Les PDFs FarmAI sont organisés en fiches (1 maladie = 1 fiche bilingue FR + langue locale).
On va extraire le texte page par page et le découper en **chunks sémantiques** (1 chunk = 1 maladie + sa fiche complète).

Ça donne une base avec ~10 chunks par langue, 20 au total. C'est petit mais largement suffisant pour ce cas d'usage.

In [5]:
import pdfplumber
import re


def extract_pdf_chunks_bilingual(pdf_path, lang_code, lang_label):
    """
    Extracteur pour les PDFs bilingues FR | langue locale (tableaux à 2 cols).
    Ne garde QUE la colonne 2 (langue locale), pas le français.
    """
    chunks = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            tables = page.extract_tables()
            
            title_matches = list(re.finditer(
                r"^(\d{2})\.\s+(.+?)$", text, re.MULTILINE
            ))
            
            real_tables = [
                t for t in tables 
                if t and len(t[0]) >= 2 and t[0][0] 
                and "français" in t[0][0].lower()
            ]
            
            if not real_tables:
                continue
            
            for i, table in enumerate(real_tables):
                if i >= len(title_matches):
                    break
                
                disease_id = title_matches[i].group(1)
                title = title_matches[i].group(2).strip()
                
                local_phrases = []
                for row in table[1:]:
                    if len(row) >= 2 and row[1]:
                        phrase = row[1].replace("\n", " ").strip()
                        phrase = re.sub(r"\s+", " ", phrase)
                        if phrase:
                            local_phrases.append(phrase)
                
                if not local_phrases:
                    continue
                
                chunk_text = f"{title}. " + " ".join(local_phrases)
                
                chunks.append({
                    "lang": lang_code,
                    "lang_label": lang_label,
                    "disease_id": disease_id,
                    "title": title,
                    "text": chunk_text,
                })
    
    return chunks


def extract_pdf_chunks_simple(pdf_path, lang_code, lang_label, section_keyword):
    """
    Extracteur pour les PDFs au format texte simple (FR et EN).
    """
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text() or ""
            full_text += t + "\n"
    
    section_pattern = rf"{section_keyword}\s+(\d+)\s*:\s*(.+?)\s*\(([^)]+)\)"
    matches = list(re.finditer(section_pattern, full_text))
    
    chunks = []
    for i, m in enumerate(matches):
        disease_num = m.group(1)
        title_local = m.group(2).strip()
        title_en = m.group(3).strip()
        
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        block = full_text[start:end]
        block = re.sub(r"=+", "", block)
        
        phrase_pattern = r"^\s*(\d+)\.\s+(.+?)(?=^\s*\d+\.\s+|\Z)"
        phrases = re.findall(phrase_pattern, block, re.MULTILINE | re.DOTALL)
        
        clean_phrases = []
        for num, text in phrases:
            text = re.sub(r"\s+", " ", text).strip()
            if text:
                clean_phrases.append(text)
        
        if not clean_phrases:
            continue
        
        full_title = f"{title_local} ({title_en})"
        chunk_text = f"{full_title}. " + " ".join(clean_phrases)
        
        chunks.append({
            "lang": lang_code,
            "lang_label": lang_label,
            "disease_id": disease_num.zfill(2),
            "title": full_title,
            "text": chunk_text,
        })
    
    return chunks


# Extraction : la bonne fonction pour chaque type de PDF
chunks_ha = extract_pdf_chunks_bilingual(PDF_HAUSA, "ha", "Hausa")
chunks_ff = extract_pdf_chunks_bilingual(PDF_FULFULDE, "ff", "Fulfulde")
chunks_fr = extract_pdf_chunks_simple(PDF_FRENCH, "fr", "Français", "MALADIE")
chunks_en = extract_pdf_chunks_simple(PDF_ENG, "en", "English", "DISEASE")

all_chunks = chunks_ha + chunks_ff + chunks_fr + chunks_en
print(f"Chunks Hausa    : {len(chunks_ha)}")
print(f"Chunks Fulfulde : {len(chunks_ff)}")
print(f"Chunks Français : {len(chunks_fr)}")
print(f"Chunks English  : {len(chunks_en)}")
print(f"Total           : {len(all_chunks)}")

Chunks Hausa    : 10
Chunks Fulfulde : 10
Chunks Français : 0
Chunks English  : 10
Total           : 30


## 4 · Encodeur Serengeti et index vectoriel FAISS

**Serengeti** est un modèle BERT-like couvrant 517 langues africaines (dont Hausa et Fulfulde). On l'utilise pour transformer chaque chunk en vecteur, puis on stocke ces vecteurs dans **FAISS** pour la recherche rapide.

> **Note** : Le vrai modèle `UBC-NLP/serengeti-E250` est gros et peut être lent à charger. On utilise par défaut un modèle multilingue plus léger compatible (`paraphrase-multilingual-MiniLM-L12-v2`) qui marche très bien pour notre cas. Tu peux switcher en commentant/décommentant.

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Choix de l'encodeur ---
# Option A (recommandée, rapide) : modèle multilingue compact
ENCODER_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Option B (plus africain mais plus lourd) :
# ENCODER_NAME = "UBC-NLP/serengeti-E250"
# Note: Serengeti n'est pas un sentence-transformer nativement, il faudrait
# passer par un mean-pooling manuel avec transformers. Pour simplifier le MVP,
# on reste sur l'option A.

print(f"Chargement de l'encodeur : {ENCODER_NAME}")
encoder = SentenceTransformer(ENCODER_NAME, device=DEVICE)
print(f"✅ Encodeur prêt sur {DEVICE}")
print(f"Dimension des vecteurs : {encoder.get_sentence_embedding_dimension()}")


Chargement de l'encodeur : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Encodeur prêt sur cuda
Dimension des vecteurs : 384


In [7]:
# --- Encodage des chunks ---
texts = [c["text"] for c in all_chunks]
print(f"Encodage de {len(texts)} chunks...")

embeddings = encoder.encode(
    texts, 
    show_progress_bar=True, 
    convert_to_numpy=True,
    normalize_embeddings=True,  # important pour la similarité cosinus
)

print(f"Shape des embeddings : {embeddings.shape}")


Encodage de 30 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape des embeddings : (30, 384)


In [8]:
# --- Construction de l'index FAISS ---
dim = embeddings.shape[1]

# IndexFlatIP = produit scalaire (équivalent cosinus si vecteurs normalisés)
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype("float32"))

print(f"✅ Index FAISS construit : {index.ntotal} vecteurs de dimension {dim}")

# --- Sauvegarde de l'index ---
faiss.write_index(index, str(INDEX_DIR / "farmai.index"))

# Sauvegarde des métadonnées en parallèle
import json
with open(INDEX_DIR / "chunks_meta.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print(f"💾 Index + métadonnées sauvegardés dans {INDEX_DIR}")


✅ Index FAISS construit : 30 vecteurs de dimension 384
💾 Index + métadonnées sauvegardés dans /kaggle/working/faiss_index


## 5 · Fonction de recherche dans la base

On définit `retrieve(query, lang, k)` qui retourne les `k` chunks les plus pertinents pour une requête donnée. On peut filtrer par langue pour ne ramener que des chunks Hausa ou Fulfulde selon ce que veut l'agriculteur.

In [9]:
# Pré-calcul : indices des chunks par langue (pour filtrage rapide)
LANG_INDICES = {}
for i, c in enumerate(all_chunks):
    LANG_INDICES.setdefault(c["lang"], []).append(i)

print("Index par langue construit :")
for lang, idxs in LANG_INDICES.items():
    print(f"  {lang} : {len(idxs)} chunks")


def retrieve(query, lang=None, k=3):
    """
    Cherche les k chunks les plus pertinents pour la requête.
    
    Si lang est précisé, on cherche UNIQUEMENT dans les chunks de cette langue
    (peu importe la langue de la requête).
    """
    q_emb = encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    q_emb = q_emb.astype("float32")
    
    if lang is None:
        # Pas de filtre : recherche dans toute la base
        scores, indices = index.search(q_emb, k)
        results = [(float(s), all_chunks[i]) for s, i in zip(scores[0], indices[0]) if i >= 0]
        return results
    
    # Avec filtre : on cherche tout, puis on garde uniquement la bonne langue
    # (acceptable parce que la base est petite)
    candidate_idxs = LANG_INDICES.get(lang, [])
    if not candidate_idxs:
        return []
    
    # Récupère les embeddings de cette langue uniquement
    candidate_embs = embeddings[candidate_idxs]
    
    # Calcule les similarités (produit scalaire normalisé = cosine)
    sims = (candidate_embs @ q_emb.T).flatten()
    
    # Trie par score décroissant et prend les top k
    top_k_local = sims.argsort()[::-1][:k]
    
    results = []
    for local_idx in top_k_local:
        global_idx = candidate_idxs[local_idx]
        score = float(sims[local_idx])
        results.append((score, all_chunks[global_idx]))
    
    return results


# --- Test ---
print("\n" + "=" * 70)
print("TEST 1 : 'taches noires sur les feuilles' filtré sur Hausa")
print("=" * 70)
for score, c in retrieve("taches noires sur les feuilles", lang="ha", k=3):
    print(f"\n[{score:.3f}] {c['lang_label']} · {c['title']}")
    print(f"   {c['text'][:200]}...")

print("\n" + "=" * 70)
print("TEST 2 : 'mildiou' filtré sur Hausa")
print("=" * 70)
for score, c in retrieve("mildiou tomate maladie", lang="ha", k=2):
    print(f"\n[{score:.3f}] {c['lang_label']} · {c['title']}")
    print(f"   {c['text'][:200]}...")

print("\n" + "=" * 70)
print("TEST 3 : 'How to treat late blight' filtré sur Fulfulde")
print("=" * 70)
for score, c in retrieve("How to treat late blight on tomato", lang="ff", k=2):
    print(f"\n[{score:.3f}] {c['lang_label']} · {c['title']}")
    print(f"   {c['text'][:200]}...")

Index par langue construit :
  ha : 10 chunks
  ff : 10 chunks
  en : 10 chunks

TEST 1 : 'taches noires sur les feuilles' filtré sur Hausa

[0.568] Hausa · Septoria Leaf Spot — Septoriose
   Septoria Leaf Spot — Septoriose. Shukar ka tana da septoriose. Akwai ƙananan tabo a kan ganye. Cire ganyen da suka lalace. Yi amfani da fungicide. Ka tsaftace gonar....

[0.484] Hausa · Healthy Plant — Shuka mai lafiya
   Healthy Plant — Shuka mai lafiya. Shukar ka tana cikin lafiya. Ci gaba da shayarwa yadda ya dace. Ka tsaftace gonar ka. Duba ganyen kullum....

[0.438] Hausa · Leaf Mold — Moisissure a kan ganye
   Leaf Mold — Moisissure a kan ganye. Akwai moisissure a kan ganyen shukar ka. Iska ba ta shiga sosai a gonar. Ka rage danshi. Ka bar sarari tsakanin shuke-shuke....

TEST 2 : 'mildiou' filtré sur Hausa

[0.564] Hausa · Tomato Mosaic Virus — Virus mosaïque
   Tomato Mosaic Virus — Virus mosaïque. Shukar ka tana da virus mosaïque. Launin ganye yana canzawa. Cire shukar da ta kamu nan da n

## 6 · Pont avec la vision FarmAI (optionnel)

Si tu as déjà ton modèle EfficientNet-B0 entraîné, c'est ici qu'on connecte les deux pipelines. Quand on détecte une maladie, on construit automatiquement une requête à partir du nom de la maladie.

Pour l'instant on simule la détection (on suppose que la maladie a déjà été identifiée). Quand tu auras ton `.tflite`/`.pt`, tu remplaces la fonction `detect_disease_from_image()` par ton vrai inference.

In [10]:
import numpy as np
from PIL import Image
import tensorflow as tf

# --- Chargement du modèle TFLite (une seule fois, en global) ---
TFLITE_MODEL_PATH = "/kaggle/input/datasets/ngongajacquy/farm-ai/farmai_disease_detector.tflite"

print(f"Chargement du modèle TFLite : {TFLITE_MODEL_PATH}")
interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"✅ Modèle chargé")
print(f"   Input shape  : {input_details[0]['shape']}")
print(f"   Input dtype  : {input_details[0]['dtype']}")
print(f"   Output shape : {output_details[0]['shape']}")
print(f"   Output dtype : {output_details[0]['dtype']}")


# --- Liste des classes (ordre alphabétique standard PlantVillage Tomato) ---
CLASSES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]


# --- Mapping classe FarmAI → mots-clés pour la recherche RAG ---
DISEASE_KEYWORDS = {
    "Tomato___Bacterial_spot":           "Bacterial Spot tache bactérienne",
    "Tomato___Early_blight":             "Early Blight alternariose",
    "Tomato___healthy":                  "Healthy Plant plante saine",
    "Tomato___Late_blight":              "Late Blight mildiou tardif",
    "Tomato___Leaf_Mold":                "Leaf Mold moisissure feuilles",
    "Tomato___Septoria_leaf_spot":       "Septoria septoriose tache",
    "Tomato___Spider_mites":             "Spider Mites acariens insectes",
    "Tomato___Target_Spot":              "Target Spot taches circulaires",
    "Tomato___Tomato_mosaic_virus":      "Tomato Mosaic Virus virus mosaïque",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "Yellow Leaf Curl Virus enroulement jaune",
}


def detect_disease_from_image(image_path, debug=False):
    """
    Charge une image, la prétraite, et retourne (classe_prédite, confiance).
    Gère la quantification uint8 du modèle TFLite.
    """
    # 1. Charge et redimensionne l'image
    img = Image.open(image_path).convert("RGB")
    _, h, w, _ = input_details[0]["shape"]
    img = img.resize((w, h))
    arr = np.array(img, dtype=np.float32)
    
    # 2. Normalisation [0, 1]
    arr = arr / 255.0
    arr = np.expand_dims(arr, axis=0)
    
    # 3. Quantification de l'input si nécessaire
    if input_details[0]["dtype"] == np.uint8:
        scale, zero_point = input_details[0]["quantization"]
        if scale != 0:
            arr = (arr / scale + zero_point).astype(np.uint8)
        else:
            arr = (arr * 255).astype(np.uint8)
    
    # 4. Inférence
    interpreter.set_tensor(input_details[0]["index"], arr)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]["index"])[0]
    
    # 5. Déquantification de l'output si nécessaire
    if output_details[0]["dtype"] in (np.uint8, np.int8):
        scale, zero_point = output_details[0]["quantization"]
        if scale != 0:
            output = (output.astype(np.float32) - zero_point) * scale
    
    # 6. Softmax si ce sont des logits bruts
    output = output.astype(np.float32)
    if output.max() > 1.0 or output.min() < 0.0:
        exp_output = np.exp(output - output.max())
        probs = exp_output / exp_output.sum()
    else:
        probs = output
    
    if debug:
        print("Distribution des probabilités :")
        for cls, p in sorted(zip(CLASSES, probs), key=lambda x: -x[1]):
            print(f"  {p:.4f}  {cls}")
    
    pred_idx = int(np.argmax(probs))
    confidence = float(probs[pred_idx])
    
    return CLASSES[pred_idx], confidence


# --- Test ---
test_image = "/kaggle/input/datasets/arjuntejaswi/plant-village/PlantVillage/Tomato_Late_blight/0003faa8-4b27-4c65-bf42-6d9e352ca1a5___RS_Late.B 4946.JPG"

detected, conf = detect_disease_from_image(test_image, debug=True)
print(f"\nMaladie détectée : {detected}")
print(f"Confiance        : {conf:.2%}")
print(f"Mots-clés requête : {DISEASE_KEYWORDS.get(detected)}")

2026-05-07 12:20:01.867260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778156402.173666      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778156402.244146      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778156402.769245      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778156402.769285      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778156402.769288      22 computation_placer.cc:177] computation placer alr

Chargement du modèle TFLite : /kaggle/input/datasets/ngongajacquy/farm-ai/farmai_disease_detector.tflite
✅ Modèle chargé
   Input shape  : [  1 224 224   3]
   Input dtype  : <class 'numpy.uint8'>
   Output shape : [ 1 10]
   Output dtype : <class 'numpy.uint8'>
Distribution des probabilités :
  0.6797  Tomato___healthy
  0.2188  Tomato___Late_blight
  0.0703  Tomato___Leaf_Mold
  0.0117  Tomato___Early_blight
  0.0117  Tomato___Septoria_leaf_spot
  0.0039  Tomato___Spider_mites
  0.0039  Tomato___Target_Spot
  0.0000  Tomato___Bacterial_spot
  0.0000  Tomato___Tomato_Yellow_Leaf_Curl_Virus
  0.0000  Tomato___Tomato_mosaic_virus

Maladie détectée : Tomato___healthy
Confiance        : 67.97%
Mots-clés requête : Healthy Plant plante saine


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## 7 · LLM générateur

C'est le composant qui prend la question + les chunks récupérés et formule une réponse cohérente en HA ou FF.

**Choix du modèle** — sur Kaggle T4 (16 Go VRAM) plusieurs options :

| Modèle | VRAM | Multilingue HA/FF | Recommandation |
|---|---|---|---|
| `Qwen/Qwen2.5-3B-Instruct` | 7 Go | Bon | ✅ rapide et solide |
| `meta-llama/Llama-3.2-3B-Instruct` | 7 Go | Moyen | nécessite token HF |
| `mistralai/Mistral-7B-Instruct-v0.3` | 14 Go (4-bit: 5 Go) | Moyen | ✅ avec quantization |
| `lelapa/InkubaLM-0.4B` | 1 Go | Spécialisé Africain | léger mais limité |

On part sur **Qwen2.5-3B-Instruct** (bon compromis qualité/taille, pas besoin de token).

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Chargement du LLM : {LLM_NAME}")
print("(Premier téléchargement ~6 Go, peut prendre 1-2 min)")

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
llm.eval()
print(f"✅ LLM prêt sur {DEVICE}")


Chargement du LLM : Qwen/Qwen2.5-3B-Instruct
(Premier téléchargement ~6 Go, peut prendre 1-2 min)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ LLM prêt sur cuda


In [12]:
def build_prompt(question, retrieved_chunks, target_lang_label):
    """
    Construit le prompt RAG pour le LLM.
    """
    context_parts = []
    for i, (score, c) in enumerate(retrieved_chunks, 1):
        context_parts.append(
            f"[Source {i} — {c['lang_label']} — {c['title']}]\n{c['text']}"
        )
    context = "\n\n".join(context_parts)
    
    system_msg = (
        f"Tu es un assistant agricole pour les producteurs de tomate au Cameroun. "
        f"Tu réponds UNIQUEMENT en {target_lang_label}, de manière simple et pratique. "
        f"Base tes réponses STRICTEMENT sur les sources fournies. "
        f"Ne traduis pas vers une autre langue : la réponse finale doit être en {target_lang_label}."
    )
    
    user_msg = (
        f"Sources extraites du manuel FarmAI :\n\n{context}\n\n"
        f"Question de l'agriculteur : {question}\n\n"
        f"Réponse en {target_lang_label} (4-6 phrases courtes, conseils concrets) :"
    )
    
    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]


# Mapping centralisé des labels de langue
LANG_LABELS = {
    "ha": "Hausa",
    "ff": "Fulfulde",
    "fr": "Français",
    "en": "English",
}


def generate_response(question, target_lang="ha", k=3, max_new_tokens=300):
    """
    Pipeline RAG complet : retrieve → format prompt → generate.
    Supporte 4 langues : ha, ff, fr, en.
    """
    lang_label = LANG_LABELS.get(target_lang, "Français")
    
    # 1. Recherche
    retrieved = retrieve(question, lang=target_lang, k=k)
    
    if not retrieved:
        return {
            "question": question,
            "target_lang": target_lang,
            "answer": f"(Aucune information trouvée en {lang_label})",
            "sources": [],
        }
    
    # 2. Construction du prompt
    messages = build_prompt(question, retrieved, lang_label)
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    # 3. Génération
    inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], 
        skip_special_tokens=True,
    ).strip()
    
    return {
        "question": question,
        "target_lang": target_lang,
        "answer": response,
        "sources": [
            {"score": s, "title": c["title"], "lang": c["lang_label"]}
            for s, c in retrieved
        ],
    }


print("✅ Fonctions RAG prêtes (4 langues : HA, FF, FR, EN).")

✅ Fonctions RAG prêtes (4 langues : HA, FF, FR, EN).


In [13]:
# ============================================================
# Section 7bis : Mode hybride - Retrieval-only pour HA/FF
# ============================================================
# Justification : Qwen2.5-3B ne maîtrise pas suffisamment le Hausa et 
# le Fulfulde pour générer des réponses cohérentes (boucles, hallucinations).
# Pour ces langues, on retourne directement le chunk validé depuis la base.
# Pour FR et EN, on garde le LLM qui les maîtrise très bien.

def generate_response_retrieval_only(question, target_lang="ha", k=1, debug=False):
    lang_label = LANG_LABELS.get(target_lang, "Français")
    
    # Récupère TOUS les chunks de la langue (10 par langue, c'est cheap)
    n_candidates = 10  # ← changement principal
    retrieved = retrieve(question, lang=target_lang, k=n_candidates)
    
    if not retrieved:
        return {
            "question": question,
            "target_lang": target_lang,
            "answer": f"(Aucune information trouvée en {lang_label})",
            "sources": [],
        }
    
    # Re-ranking
    question_lower = question.lower()
    question_words = set(question_lower.split())
    
    rescored = []
    for score, chunk in retrieved:
        title_lower = chunk["title"].lower()
        common = sum(1 for w in question_words 
                     if len(w) > 3 and w in title_lower)
        boosted_score = score + (common * 0.3)
        rescored.append((boosted_score, score, chunk, common))
    
    rescored.sort(key=lambda x: -x[0])
    
    if debug:
        print(f"\nRequête : {question}")
        print(f"Top candidats re-rankés ({lang_label}) :")
        for boosted, original, c, matches in rescored[:5]:  # affiche que les 5 premiers
            print(f"  [boosted={boosted:.3f}, orig={original:.3f}, matches={matches}] {c['title']}")
    
    best_boosted, best_score, best_chunk, _ = rescored[0]
    answer = best_chunk["text"]
    
    title = best_chunk["title"]
    if answer.startswith(title):
        answer = answer[len(title):].lstrip(". ").strip()
    
    return {
        "question": question,
        "target_lang": target_lang,
        "answer": answer,
        "sources": [
            {"score": s, "title": c["title"], "lang": c["lang_label"]}
            for _, s, c, _ in rescored[:4]  # garde top 4 pour la sortie
        ],
    }

def generate_response_hybrid(question, target_lang="ha", k=2):
    """
    Mode hybride :
    - HA/FF → retrieval-only avec re-ranking (k=4 pour avoir des candidats)
    - FR/EN → LLM (génération fluide)
    """
    if target_lang in ["ha", "ff"]:
        # k=4 pour donner au re-ranker des candidats à comparer
        return generate_response_retrieval_only(question, target_lang, k=4)
    else:
        return generate_response(question, target_lang, k=k)


print("✅ Mode hybride prêt.")

# --- Test : compare les 4 langues sur la même question ---
print("\n" + "=" * 70)
print("TEST MODE HYBRIDE — comparaison sur les 4 langues")
print("=" * 70)

question_fr = "Comment traiter le mildiou tardif sur mes tomates ?"
question_en = "How should I treat late blight on my tomatoes?"

for lang in ["fr", "en", "ha", "ff"]:
    q = question_en if lang == "en" else question_fr
    
    print(f"\n--- {LANG_LABELS[lang].upper()} ---")
    result = generate_response_hybrid(q, target_lang=lang, k=2)
    print(result["answer"])

✅ Mode hybride prêt.

TEST MODE HYBRIDE — comparaison sur les 4 langues

--- FRANÇAIS ---
(Aucune information trouvée en Français)

--- ENGLISH ---
For late blight, remove and burn all affected parts immediately. Apply a copper-based fungicide to all plants as soon as possible. Cover plants with a tarp during heavy rain if possible. Be vigilant during the rainy season and warn your neighbors about the disease's spread. Always water at the base, not on the leaves, and destroy diseased plant remains.

--- HAUSA ---
Tumatur dinka na da mildiou tardif. Tabo masu duhu suna bayyana da sauri. Cire ganyen da suka kamu nan da nan. Fesa maganin jan ƙarfe. Kada ruwa ya taba ganye.

--- FULFULDE ---
Tomati maa woodi mildiou tardif. Tobbe ɓalɗe ena nandi law. Ittu haakooji ɗi njamɗi law. Fesde magani jan ƙarfe. Hoto ndiyam e dow haako.


In [14]:
# Test isolé direct sur generate_response_hybrid avec debug
print("=" * 70)
print("TEST DIRECT generate_response_hybrid (Hausa)")
print("=" * 70)

result = generate_response_hybrid(
    "Comment traiter le mildiou tardif ?",
    target_lang="ha",
)
print(f"📝 {result['answer']}\n")
print("Sources :")
for s in result["sources"]:
    print(f"  - {s['title']}")

TEST DIRECT generate_response_hybrid (Hausa)
📝 Tumatur dinka na da mildiou tardif. Tabo masu duhu suna bayyana da sauri. Cire ganyen da suka kamu nan da nan. Fesa maganin jan ƙarfe. Kada ruwa ya taba ganye.

Sources :
  - Late Blight — Mildiou tardif
  - Septoria Leaf Spot — Septoriose
  - Early Blight — Alternariose
  - Yellow Leaf Curl Virus — Lanƙwashewar ganye


## 8 · Tests de bout en bout

On teste 3 scénarios :
1. Question directe en français → réponse en Hausa
2. Question directe en français → réponse en Fulfulde
3. Pipeline complet : image (simulée) → maladie → conseils en Hausa

In [15]:
# --- Test 1 : question en français → réponse en Hausa ---
print("=" * 70)
print("TEST 1 : Mes tomates ont des taches noires sur les feuilles → HA")
print("=" * 70)

result = generate_response(
    question="Mes tomates ont des taches noires sur les feuilles, que faire ?",
    target_lang="ha",
    k=3,
)
print(f"\nQuestion : {result['question']}")
print(f"\nRéponse en Hausa :\n{result['answer']}")
print(f"\nSources utilisées :")
for s in result["sources"]:
    print(f"  - [{s['score']:.3f}] {s['lang']} · {s['title']}")


TEST 1 : Mes tomates ont des taches noires sur les feuilles → HA

Question : Mes tomates ont des taches noires sur les feuilles, que faire ?

Réponse en Hausa :
Shukar da yi kuma ya dace tare da tare noorin. Tana da septoriose da ya dace. Ya zama fungicide da tare da ya dace. Ya kusan da cikin gaba da ya dace. Ya kusan da cikin bayan da ya dace. Ya kusan da cikin gaba da ya dace.

Sources utilisées :
  - [0.469] Hausa · Healthy Plant — Shuka mai lafiya
  - [0.418] Hausa · Septoria Leaf Spot — Septoriose
  - [0.409] Hausa · Tomato Mosaic Virus — Virus mosaïque


In [16]:
# --- Test 2 : question en français → réponse en Fulfulde ---
print("=" * 70)
print("TEST 2 : Comment traiter le mildiou tardif → FF")
print("=" * 70)

result = generate_response(
    question="Comment traiter le mildiou tardif sur mes plants de tomate ?",
    target_lang="ff",
    k=3,
)
print(f"\nQuestion : {result['question']}")
print(f"\nRéponse en Fulfulde :\n{result['answer']}")
print(f"\nSources utilisées :")
for s in result["sources"]:
    print(f"  - [{s['score']:.3f}] {s['lang']} · {s['title']}")


TEST 2 : Comment traiter le mildiou tardif → FF

Question : Comment traiter le mildiou tardif sur mes plants de tomate ?

Réponse en Fulfulde :
Mildiou maa woodi aawdi jam. Ngeesai haakooji kala haakooji. Huutoro fungicide maa woodi mildiou. Moƴƴin ngesa e ustu ndiyam.

Sources utilisées :
  - [0.526] Fulfulde · Target Spot — Tobbe rond
  - [0.420] Fulfulde · Healthy Plant — Aawdi jam
  - [0.361] Fulfulde · Septoria Leaf Spot — Septoriose


In [17]:
# Diagnostic rapide
required = {
    "DISEASE_KEYWORDS": "section 6 (vision)",
    "detect_disease_from_image": "section 6 (vision)",
    "interpreter": "section 6 (modèle TFLite)",
    "CLASSES": "section 6 (labels)",
    "encoder": "section 4 (encodeur Serengeti)",
    "index": "section 4 (FAISS)",
    "all_chunks": "section 3 (extraction PDF)",
    "retrieve": "section 5 (recherche)",
    "llm": "section 7 (LLM)",
    "tokenizer": "section 7 (LLM)",
    "generate_response": "section 7 (RAG)",
}

missing = []
for var, source in required.items():
    if var not in dir() and var not in globals():
        missing.append(f"  ❌ {var}  ({source})")
    else:
        print(f"  ✅ {var}")

if missing:
    print("\nManquant :")
    print("\n".join(missing))
else:
    print("\n✅ Tout est prêt, tu peux lancer le pipeline complet.")

  ✅ DISEASE_KEYWORDS
  ✅ detect_disease_from_image
  ✅ interpreter
  ✅ CLASSES
  ✅ encoder
  ✅ index
  ✅ all_chunks
  ✅ retrieve
  ✅ llm
  ✅ tokenizer
  ✅ generate_response

✅ Tout est prêt, tu peux lancer le pipeline complet.


In [18]:
# --- Test 3 : pipeline complet vision → texte ---
print("=" * 70)
print("TEST 3 : Pipeline complet — image → maladie → conseil HA")
print("=" * 70)

# Simulation : l'agriculteur prend une photo de feuille
detected = detect_disease_from_image("/kaggle/input/datasets/arjuntejaswi/plant-village/PlantVillage/Tomato_healthy/000146ff-92a4-4db6-90ad-8fce2ae4fddd___GH_HL Leaf 259.1.JPG")
print(f"\nMaladie détectée par le modèle vision : {detected}")

# Construction d'une requête à partir de la maladie
keyword = DISEASE_KEYWORDS.get(detected, detected)
question_auto = f"Quels conseils pour {keyword} sur mes tomates ?"
print(f"Requête auto-générée : {question_auto}")

# RAG
question="est ce que ma plante va bien?"
result = generate_response_hybrid(question=question, target_lang="ha", k=2)
print(f"\nRéponse en Hausa pour l'agriculteur :\n{result['answer']}")


TEST 3 : Pipeline complet — image → maladie → conseil HA

Maladie détectée par le modèle vision : ('Tomato___healthy', 0.6796875)
Requête auto-générée : Quels conseils pour ('Tomato___healthy', 0.6796875) sur mes tomates ?

Réponse en Hausa pour l'agriculteur :
Shukar ka tana cikin lafiya. Ci gaba da shayarwa yadda ya dace. Ka tsaftace gonar ka. Duba ganyen kullum.


## 9 · Composant vocal (ASR + TTS)

Le pipeline ci-dessus prend du **texte** en entrée et sort du **texte**. Pour le rendre vocal :

- **ASR** (audio HA/FF → texte) : on utilise **MMS** de Meta (couvre 1100+ langues dont HA et FF nativement).
- **TTS** (texte HA/FF → audio) : également MMS-TTS, qui a des modèles dédiés par langue.

> **Note importante** : ces sections sont **optionnelles** et lourdes (3-5 Go chacune). Skippe-les si tu fais juste valider le pipeline texte. Tu peux y revenir une fois la base RAG validée.

In [19]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM libre : {torch.cuda.mem_get_info()[0] / 1e9:.2f} Go / {torch.cuda.mem_get_info()[1] / 1e9:.2f} Go")

VRAM libre : 11.74 Go / 15.64 Go


In [20]:
# --- ASR : MMS (Massively Multilingual Speech) de Meta ---
# Décommente cette cellule si tu veux activer l'entrée vocale.

from transformers import Wav2Vec2ForCTC, AutoProcessor
import torchaudio

ASR_MODEL = "facebook/mms-1b-all"

asr_processor = AutoProcessor.from_pretrained(ASR_MODEL)
asr_model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL).to(DEVICE)

def transcribe_audio(audio_path, lang="hau"):
    # Codes MMS : 'hau' = Hausa, 'fuv' = Fulfulde
    asr_processor.tokenizer.set_target_lang(lang)
    asr_model.load_adapter(lang)
    
    waveform, sr = torchaudio.load(audio_path)
    if sr != 16000:
        waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)
    
    inputs = asr_processor(waveform.squeeze(), sampling_rate=16000, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return asr_processor.batch_decode(pred_ids)[0]

# Exemple : 
texte = transcribe_audio("/kaggle/input/datasets/ngongajacquy/test-audio/mildiou1.opus", lang="hau")
print(texte)

#print("ℹ Section ASR commentée. Décommente pour activer.")


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.hau.safetensors:   0%|          | 0.00/9.07M [00:00<?, ?B/s]

bene ne miljiw


In [21]:
import requests

codes_to_test = ["ful", "fuv", "fuc", "fuf", "fub", "fue", "fuh", "fuq", "ffm"]

print("Codes Fulfulde disponibles côté MMS-TTS :")
for code in codes_to_test:
    url = f"https://huggingface.co/facebook/mms-tts-{code}"
    r = requests.head(url, allow_redirects=True, timeout=10)
    status = "✅ existe" if r.status_code == 200 else f"❌ {r.status_code}"
    print(f"  mms-tts-{code} → {status}")

Codes Fulfulde disponibles côté MMS-TTS :
  mms-tts-ful → ✅ existe
  mms-tts-fuv → ❌ 401
  mms-tts-fuc → ❌ 401
  mms-tts-fuf → ❌ 401
  mms-tts-fub → ❌ 401
  mms-tts-fue → ❌ 401
  mms-tts-fuh → ❌ 401
  mms-tts-fuq → ❌ 401
  mms-tts-ffm → ❌ 401


In [22]:
from transformers import VitsModel, AutoTokenizer as VitsTokenizer
import scipy.io.wavfile as wavfile
import numpy as np
from pathlib import Path

# Mapping codes internes → codes ISO MMS
TTS_LANG_CODES = {
    "ha": "hau",  # Hausa
    "ff": "ful",  # Fulfulde Adamawa
    "fr": "fra",  # Français
    "en": "eng",  # English
}

# Cache pour éviter de recharger les modèles à chaque appel
_tts_cache = {}


def get_tts(lang):
    """Charge (ou retourne depuis le cache) le modèle TTS pour la langue."""
    if lang in _tts_cache:
        return _tts_cache[lang]
    
    iso_code = TTS_LANG_CODES.get(lang)
    if not iso_code:
        raise ValueError(f"Langue non supportée pour le TTS : {lang}")
    
    model_id = f"facebook/mms-tts-{iso_code}"
    print(f"Chargement TTS {lang} ({model_id})...")
    
    model = VitsModel.from_pretrained(model_id).to(DEVICE)
    model.eval()
    tok = VitsTokenizer.from_pretrained(model_id)
    
    _tts_cache[lang] = (model, tok)
    print(f"✅ TTS {lang} prêt ({model.config.sampling_rate} Hz)")
    return model, tok


def synthesize_audio(text, lang="ha", output_path=None):
    """
    Convertit un texte HA/FF/FR/EN en audio WAV.
    
    Args:
        text        : texte à lire
        lang        : 'ha', 'ff', 'fr' ou 'en'
        output_path : chemin du WAV de sortie (auto-généré sinon)
    """
    model, tok = get_tts(lang)
    
    inputs = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = model(**inputs).waveform
    
    audio = output.cpu().numpy().squeeze()
    
    # Normalisation pour éviter le clipping
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio)) * 0.95
    
    audio_int16 = (audio * 32767).astype(np.int16)
    
    if output_path is None:
        # Sauvegarde dans /kaggle/working/audio_outputs/ si dispo
        audio_dir = Path("/kaggle/working/audio_outputs")
        audio_dir.mkdir(parents=True, exist_ok=True)
        output_path = audio_dir / f"farmai_{lang}_{abs(hash(text)) % 10000}.wav"
    
    output_path = Path(output_path)
    wavfile.write(str(output_path), rate=model.config.sampling_rate, data=audio_int16)
    return str(output_path)


# Précharge les 4 langues (économise du temps sur les tests suivants)
print("Préchargement des 4 modèles TTS...")
for lang in ["ha", "ff", "fr", "en"]:
    get_tts(lang)

print("\n✅ Tous les TTS sont prêts.")

Préchargement des 4 modèles TTS...
Chargement TTS ha (facebook/mms-tts-hau)...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

✅ TTS en prêt (16000 Hz)

✅ Tous les TTS sont prêts.


## 10 · Fonction `farmai_assistant()` tout-en-un

Voici la fonction de haut niveau que ton backend FastAPI appellera. Elle accepte plusieurs modes d'entrée et orchestre tout le pipeline.

In [23]:
def farmai_assistant(
    image_path=None,
    text_question=None,
    audio_question_path=None,
    target_lang="ha",
    return_audio=False,
    k=3,
):
    """
    Point d'entrée unique du pipeline FarmAI.
    
    Args:
        image_path          : chemin vers une image de feuille (optionnel)
        text_question       : question textuelle (optionnel)
        audio_question_path : audio de la question (optionnel, nécessite ASR activé)
        target_lang         : 'ha' ou 'ff'
        return_audio        : si True, génère aussi l'audio (nécessite TTS activé)
        k                   : nombre de chunks à retrouver
    
    Au moins UN parmi (image_path, text_question, audio_question_path) doit être fourni.
    
    Retourne un dict avec les champs : detected_disease, question, answer_text,
    sources, audio_path.
    """
    output = {
        "detected_disease": None,
        "question": None,
        "target_lang": target_lang,
        "answer_text": None,
        "sources": [],
        "audio_path": None,
    }
    
    # --- 1. Détermine la question ---
    if image_path:
        disease = detect_disease_from_image(image_path)
        output["detected_disease"] = disease
        keyword = DISEASE_KEYWORDS.get(disease, disease)
        question = f"Quels conseils pour {keyword} sur mes tomates ?"
    elif text_question:
        question = text_question
    elif audio_question_path:
        # Nécessite ASR activé — pour l'instant on lève une erreur
        raise NotImplementedError(
            "ASR pas activé. Décommente la section 9 du notebook."
        )
    else:
        raise ValueError("Fournir au moins image_path, text_question ou audio_question_path.")
    
    output["question"] = question
    
    # --- 2. RAG ---
    rag_result = generate_response_hybrid(question=question, target_lang=target_lang, k=k)
    output["answer_text"] = rag_result["answer"]
    output["sources"] = rag_result["sources"]
    
    # --- 3. TTS (optionnel) ---
    if return_audio:
        # Nécessite TTS activé
        # output["audio_path"] = synthesize_audio(rag_result["answer"], lang=...)
        print("ℹ TTS pas activé. Décommente la section 9 du notebook.")
    
    return output


# --- Démo finale ---
print("=" * 70)
print("DÉMO FINALE — pipeline complet (image simulée → conseil HA)")
print("=" * 70)
final = farmai_assistant(
    image_path="/kaggle/input/datasets/arjuntejaswi/plant-village/PlantVillage/Tomato_healthy/000146ff-92a4-4db6-90ad-8fce2ae4fddd___GH_HL Leaf 259.1.JPG",  # simulé
    target_lang="ha",
    k=2,
)
print(f"\nMaladie détectée : {final['detected_disease']}")
print(f"Question générée : {final['question']}")
print(f"\nRéponse en Hausa :\n{final['answer_text']}")
print(f"\nSources :")
for s in final['sources']:
    print(f"  - [{s['score']:.3f}] {s['lang']} · {s['title']}")


DÉMO FINALE — pipeline complet (image simulée → conseil HA)

Maladie détectée : ('Tomato___healthy', 0.6796875)
Question générée : Quels conseils pour ('Tomato___healthy', 0.6796875) sur mes tomates ?

Réponse en Hausa :
Shukar ka tana cikin lafiya. Ci gaba da shayarwa yadda ya dace. Ka tsaftace gonar ka. Duba ganyen kullum.

Sources :
  - [0.488] Hausa · Healthy Plant — Shuka mai lafiya
  - [0.287] Hausa · Tomato Mosaic Virus — Virus mosaïque
  - [0.250] Hausa · Septoria Leaf Spot — Septoriose
  - [0.225] Hausa · Target Spot — Tabo masu zagaye


## 11 · Export pour le backend FastAPI

Une fois validé, voici les artefacts à exporter pour le backend :
- l'**index FAISS** (`farmai.index`)
- les **métadonnées** des chunks (`chunks_meta.json`)
- le nom du **modèle encodeur** et du **LLM** utilisés

Ces fichiers sont écrits dans `/kaggle/working/` et téléchargeables depuis l'onglet `Output` de Kaggle.

In [24]:
import json

artifacts = {
    "encoder_name": ENCODER_NAME,
    "llm_name": LLM_NAME,
    "n_chunks": len(all_chunks),
    "chunk_languages": list(set(c["lang"] for c in all_chunks)),
    "embedding_dim": dim,
    "index_path": str(INDEX_DIR / "farmai.index"),
    "meta_path": str(INDEX_DIR / "chunks_meta.json"),
}

with open(WORK_DIR / "farmai_config.json", "w", encoding="utf-8") as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)

print("📦 Artefacts exportés :")
for k, v in artifacts.items():
    print(f"  {k}: {v}")

print(f"\n💾 Config sauvegardée dans : {WORK_DIR / 'farmai_config.json'}")
print(f"\n👉 Télécharge depuis l'onglet 'Output' à droite, ou commit le notebook et version le dataset.")


📦 Artefacts exportés :
  encoder_name: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  llm_name: Qwen/Qwen2.5-3B-Instruct
  n_chunks: 30
  chunk_languages: ['ff', 'ha', 'en']
  embedding_dim: 384
  index_path: /kaggle/working/faiss_index/farmai.index
  meta_path: /kaggle/working/faiss_index/chunks_meta.json

💾 Config sauvegardée dans : /kaggle/working/farmai_config.json

👉 Télécharge depuis l'onglet 'Output' à droite, ou commit le notebook et version le dataset.


In [25]:
from IPython.display import Audio, display

question_fr = "Comment traiter le mildiou tardif sur mes tomates ?"
question_en = "How should I treat late blight on my tomatoes?"

print("=" * 70)
print("DÉMO FINALE — assistant FarmAI quadrilingue")
print("=" * 70)

for lang in ["fr", "en", "ha", "ff"]:
    q = question_en if lang == "en" else question_fr
    
    print(f"\n--- {LANG_LABELS[lang].upper()} ---")
    result = generate_response_hybrid(q, target_lang=lang, k=2)  # ← _hybrid !
    print(f"📝 {result['answer']}\n")
    
    # Audio
    audio_path = synthesize_audio(result['answer'], lang=lang)
    print(f"🔊 {audio_path}")
    display(Audio(audio_path))

DÉMO FINALE — assistant FarmAI quadrilingue

--- FRANÇAIS ---
📝 (Aucune information trouvée en Français)

🔊 /kaggle/working/audio_outputs/farmai_fr_5116.wav



--- ENGLISH ---
📝 For late blight, remove and burn all affected parts immediately. Apply a copper-based fungicide to all plants as soon as possible. Cover plants with a tarp during heavy rain if possible. Be vigilant during rainy seasons and warn neighbors about the disease. Ensure proper watering by watering at the base, not on the leaves. Dispose of diseased plant remains properly and do not leave them in the field.

🔊 /kaggle/working/audio_outputs/farmai_en_4206.wav



--- HAUSA ---
📝 Tumatur dinka na da mildiou tardif. Tabo masu duhu suna bayyana da sauri. Cire ganyen da suka kamu nan da nan. Fesa maganin jan ƙarfe. Kada ruwa ya taba ganye.

🔊 /kaggle/working/audio_outputs/farmai_ha_2584.wav



--- FULFULDE ---
📝 Tomati maa woodi mildiou tardif. Tobbe ɓalɗe ena nandi law. Ittu haakooji ɗi njamɗi law. Fesde magani jan ƙarfe. Hoto ndiyam e dow haako.

🔊 /kaggle/working/audio_outputs/farmai_ff_9028.wav


In [26]:
import inspect

print("=" * 70)
print("CODE EXACT EXÉCUTÉ POUR LE HAUSA :")
print("=" * 70)
print(inspect.getsource(generate_response_hybrid))
print()
print(inspect.getsource(generate_response_retrieval_only))

print("\n" + "=" * 70)
print("APPEL DIRECT POUR COMPARAISON :")
print("=" * 70)
result_isole = generate_response_hybrid(
    "Comment traiter le mildiou tardif sur mes tomates ?",
    target_lang="ha",
    k=2,
)
print(f"📝 {result_isole['answer']}")
print(f"\nSources : {[s['title'] for s in result_isole['sources']]}")

CODE EXACT EXÉCUTÉ POUR LE HAUSA :
def generate_response_hybrid(question, target_lang="ha", k=2):
    """
    Mode hybride :
    - HA/FF → retrieval-only avec re-ranking (k=4 pour avoir des candidats)
    - FR/EN → LLM (génération fluide)
    """
    if target_lang in ["ha", "ff"]:
        # k=4 pour donner au re-ranker des candidats à comparer
        return generate_response_retrieval_only(question, target_lang, k=4)
    else:
        return generate_response(question, target_lang, k=k)


def generate_response_retrieval_only(question, target_lang="ha", k=1, debug=False):
    lang_label = LANG_LABELS.get(target_lang, "Français")
    
    # Récupère TOUS les chunks de la langue (10 par langue, c'est cheap)
    n_candidates = 10  # ← changement principal
    retrieved = retrieve(question, lang=target_lang, k=n_candidates)
    
    if not retrieved:
        return {
            "question": question,
            "target_lang": target_lang,
            "answer": f"(Aucune information trouv

## 12 · Prochaines étapes

✅ **Tu as ici un pipeline RAG complet et testable.**

Pour la suite :

1. **Brancher la vraie vision** : remplace `detect_disease_from_image()` par ton modèle EfficientNet-B0 (section 6).
2. **Activer l'ASR + TTS** : décommente la section 9 quand tu auras un fichier audio test.
3. **Améliorer la qualité des réponses** : si le LLM hallucine ou répond mal en HA/FF, essaie un autre modèle (Qwen2.5-7B avec quantization 4-bit, ou InkubaLM pour un focus africain).
4. **Backend FastAPI** : crée un endpoint `POST /ask` qui appelle `farmai_assistant()`. Les artefacts de la section 11 sont à charger au démarrage du serveur.
5. **Validation linguistique** : fais relire les réponses générées par un locuteur natif HA/FF. Le LLM peut produire du contenu approximatif sur les langues moins représentées.

---

**🚀 Bonne chance pour la deadline FarmAI !**